# Đề tài 5KH · Dự đoán đầu ra Python qua nhiều tệp

**Chạy trên Jupyter hoặc Colab** · Chọn Python 3 rồi bấm **Run All**. Bản này cấu hình sẵn Qwen3-VL-8B-Instruct-FP8. Nhập khóa API VTS qua ô nhập ẩn hoặc đặt biến môi trường VTS_LLM_API_KEY. Đặt **RUN_LLM = False** để thử dữ liệu và kiểm tra prompt mà không gọi API.

Mặc định notebook chạy **12 ca tổng hợp mới**, mỗi ca có **9 cấu hình đọc mã và 1 phép thử số học độc lập**. Đặt **CASES_PER_FAMILY = 1** để kiểm tra kết nối bằng 30 yêu cầu. Tăng số ca sau khi kiểm tra thử và chốt trước cấu hình. Kết quả được lưu từng yêu cầu vào ket_qua_5KH_v3; có thể chạy tiếp từ log.

| Câu hỏi | Đối chứng | Phép đo |
|---|---|---|
| RQ1 · Vị trí mã | Giữ cùng đoạn mã, độ dài và thứ tự mã nhiễu. Dịch nhóm hàm phụ thuộc đến đầu, giữa hoặc cuối. | Khớp chuỗi gọi và khớp đầu ra. |
| RQ2 · Chọn và sắp xếp mã | BM25 và BM25 + đồ thị đều được chạy với thứ tự xếp hạng và thứ tự chuẩn theo định danh. Trong mỗi phương pháp, hai thứ tự có cùng tập đoạn mã. | Tỷ lệ tìm đủ hàm, khớp chuỗi gọi và khớp đầu ra. |
| RQ3 · Bước sai đầu tiên | Oracle cấp đủ hàm đã thực thi và yêu cầu vết gọi có cấu trúc. Có thêm oracle chỉ yêu cầu đáp án và phép thử biểu thức độc lập. | Đối chiếu từng lần vào hàm, đối số, lần trả về, giá trị trả về và đáp án cuối. |

**Thiết kế nghiên cứu.** So sánh ghép cặp trên cùng ca và mô hình, dùng Python làm chuẩn kiểm chứng. Ghi cấu hình, mọi prompt, thứ tự đoạn mã, đáp án và phản hồi thô để tái lập. Dữ liệu tổng hợp cho phép kiểm soát các can thiệp. Các ca cùng họ dùng chung cấu trúc; độ tin cậy bên ngoài cần được kiểm chứng trên kho Python công khai.

**Giới hạn diễn giải.** Vết gọi do mô hình báo cáo là một đầu ra có thể kiểm tra. Đối chứng oracle vừa bỏ mã nhiễu vừa cấp đủ hàm. Phép thử số học chỉ cung cấp phép tính cho lời gọi fold đầu tiên và được chấm riêng; nó không cung cấp thông tin trong các prompt đọc mã.

**Từ khóa** · LLM code understanding, output prediction, call-graph retrieval, program tracing, lost in the middle.

Nguồn phương pháp · [Sense and Sensitivity](https://arxiv.org/abs/2505.13353v5), [RepoReasoner](https://arxiv.org/abs/2607.25996), [Code Comprehensor](https://arxiv.org/abs/2610.30016v2), [Evidence-Centered Survey](https://arxiv.org/abs/2608.21107v1).

In [ ]:
import ast
import csv
import getpass
import hashlib
import importlib
import importlib.util
import importlib.metadata
import json
import math
import os
import random
import re
import subprocess
import sys
import time
import types
from collections import Counter, defaultdict
from pathlib import Path

from IPython.display import Markdown, display

if importlib.util.find_spec("tiktoken") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tiktoken==0.12.0"])
    importlib.invalidate_caches()
import requests
import tiktoken

SEED = 544015
CASES_PER_FAMILY = 4
DISTRACTOR_MODULES = 64
RETRIEVAL_TOKEN_CAP = 300
TOKENIZER = "o200k_base"       # Ngân sách token tham chiếu, khác tokenizer phía Qwen.
RUN_LLM = True
VTS_CHAT_COMPLETIONS_URL = "https://llm-node3.vts.io.vn/v1/chat/completions"
CHAT_COMPLETIONS_URL = os.environ.get("LLM_CHAT_COMPLETIONS_URL", VTS_CHAT_COMPLETIONS_URL)
MODEL = os.environ.get("LLM_MODEL", "Qwen/Qwen3-VL-8B-Instruct-FP8")
MAX_OUTPUT_TOKENS = 1000
OUTPUT_LIMIT_FIELD = "max_tokens"  # Một số máy chủ dùng "max_completion_tokens".
TEMPERATURE = 0
OUTPUT_DIR = Path("ket_qua_5KH_v3")

encoder = tiktoken.get_encoding(TOKENIZER)

def token_count(text):
    return len(encoder.encode_ordinary(text))

assert CASES_PER_FAMILY >= 1 and DISTRACTOR_MODULES >= 8 and RETRIEVAL_TOKEN_CAP > 0

## 1. Dữ liệu và đáp án từ thực thi

Notebook chỉ chạy mã Python do bộ sinh bên dưới tạo ra. Trình ghi vết lưu từng lần vào và ra hàm theo thứ tự xảy ra, kể cả khi cùng một hàm được gọi nhiều lần. Mỗi lần gọi có tên đầy đủ, đối số và giá trị trả về. Tập hàm bắt buộc cho phép đo truy xuất được lấy từ vết thực thi và bỏ các hàm trùng.

In [ ]:
FAMILIES = ("pipeline", "nested", "conditional")

def make_case(family, index, rng):
    a, b, c, d = (rng.randint(11, 79) for _ in range(4))
    x = rng.randint(4, 80) * 2 + index % 2
    if family == "pipeline":
        left = f"    return calibration.offset(value + {a}) * 2"
        right = f"    return calibration.offset(value - {b}) + {c}"
    elif family == "nested":
        left = f"    return calibration.wrap(calibration.offset(value) - {a})"
        right = f"    return calibration.wrap(value + {b}) * 3"
    else:
        left = (f"    if value % 3 == 0:\n"
                f"        return calibration.wrap(value + {a})\n"
                f"    return calibration.offset(value - {b})")
        right = f"    return calibration.wrap(value - {c}) + calibration.offset(value)"
    fold_call = ("getattr(rounding_profile, 'fold')" if family == "nested"
                 else "rounding_profile.fold")
    sources = {
        "entry": (
            "import pricing_routes\n\n"
            "def compute(value):\n"
            "    if value % 2 == 0:\n"
            "        return pricing_routes.primary(value)\n"
            "    return pricing_routes.secondary(value)\n"
        ),
        "pricing_routes": (
            f"import calibration\n\ndef primary(value):\n{left}\n\n"
            f"def secondary(value):\n{right}\n"
        ),
        "calibration": (
            f"import rounding_profile\n\n"
            f"def offset(value):\n    return {fold_call}(value + {d})\n\n"
            f"def wrap(value):\n    return {fold_call}(value * 2 - {b})\n"
        ),
        "rounding_profile": f"def fold(value):\n    return (value * {a} + {c}) % {101 + d}\n",
    }
    # Các đoạn nhiễu dùng thuật ngữ và định danh gần với truy vấn.
    for number in range(DISTRACTOR_MODULES):
        delta = rng.randint(1, 93)
        sources[f"extras_{number:02}"] = (
            f"def primary(value):\n"
            f"    adjusted = (value + {delta}) * {2 + number}\n"
            f"    return adjusted % {103 + number}\n\n"
            f"def secondary(value):\n"
            f"    adjusted = (value - {delta}) * {3 + number}\n"
            f"    return adjusted % {127 + number}\n"
        )
    return {"id": f"{family}-{index:03}", "family": family, "x": x,
            "sources": sources, "fold_coefficients": [a, c, 101 + d]}

def execute_fixture(case):
    loaded = {}
    real_import = __import__

    def fixture_import(name, globals=None, locals=None, fromlist=(), level=0):
        if not level and name in case["sources"]:
            return load_module(name)
        return real_import(name, globals, locals, fromlist, level)

    def load_module(name):
        if name not in loaded:
            mod = types.ModuleType(name)
            loaded[name] = mod
            mod.__dict__["__builtins__"] = {**vars(importlib.import_module("builtins")), "__import__": fixture_import}
            filename = f"<fixture:{name}>"
            exec(compile(case["sources"][name], filename, "exec"), mod.__dict__)
        return loaded[name]

    entry = load_module("entry")
    calls, events, active_frames = [], [], {}

    def capture(frame, event, arg):
        filename = frame.f_code.co_filename
        if not filename.startswith("<fixture:"):
            return
        if event == "call":
            module = filename.removeprefix("<fixture:").removesuffix(">")
            index = len(calls)
            active_frames[frame] = index
            calls.append({"function": f"{module}.{frame.f_code.co_name}",
                          "arg": frame.f_locals["value"]})
            events.append(("call", index))
        elif event == "return":
            index = active_frames.pop(frame)
            calls[index]["result"] = arg
            events.append(("return", index))

    previous = sys.getprofile()
    sys.setprofile(capture)
    try:
        answer = entry.compute(case["x"])
    finally:
        sys.setprofile(previous)
    return answer, calls, events

rng = random.Random(SEED)
cases = [make_case(family, i, rng) for family in FAMILIES for i in range(CASES_PER_FAMILY)]
for case in cases:
    case["answer"], case["calls"], case["events"] = execute_fixture(case)
    case["trace"] = list(dict.fromkeys(call["function"] for call in case["calls"]))
    assert case["trace"][0] == "entry.compute"
    assert case["calls"][0]["result"] == case["answer"]
    assert all(type(call["arg"]) is int and type(call["result"]) is int for call in case["calls"])

print(f"{len(cases)} ca, {len(FAMILIES)} họ chương trình, {len(cases[0]['sources'])} tệp mỗi ca.")
print("Ví dụ vết chạy:", " → ".join(cases[0]["trace"]))

## 2. Truy xuất và dựng các đối chứng

BM25 cộng các thành phần bằng math.fsum, làm tròn điểm ở 12 chữ số thập phân và phá đồng hạng bằng định danh hàm. Đồ thị tĩnh ưu tiên các hàm tìm được trong tối đa ba cạnh rồi điền phần còn lại theo BM25. BM25 và đồ thị có cùng giới hạn token. Với mỗi phương pháp, chốt tập đoạn còn vừa ngân sách ở cả hai thứ tự trước khi gọi mô hình. Thứ tự chuẩn sắp theo định danh module.hàm. Chỉ thứ tự xếp hạng thay đổi giữa hai cấu hình dùng chung tập đoạn.

Cả ba vị trí RQ1 dùng chung thứ tự gây nhiễu chuẩn, độc lập với điểm BM25. Mỗi ca được kiểm tra độ dài prompt chính xác bằng tokenizer tham chiếu và kiểm tra tập đoạn trước khi gửi. Đếm số token thực tế của máy chủ từ trường usage nếu có.

In [ ]:
ENTRY = "entry.compute"
SYSTEM = "Evaluate Python code using only the supplied source and input."
TRACE_ASK = (
    "Return a single compact JSON object. Put the exact output in integer field answer first. "
    "Then put all invoked functions, including entry.compute, in calls in order of invocation. "
    "For each invocation give its fully qualified function name, integer arg, and integer result. "
    'Schema example: {"answer": 3, "calls": [{"function": "entry.compute", "arg": 1, "result": 3}]}. '
    "Keep repeated calls in the list and omit calls on unexecuted branches. Do not include extra text."
)
DIRECT_ASK = 'Return one JSON object with a single field "answer" containing the final integer.'
CONDITION_NAMES = (
    "position_front", "position_middle", "position_end", "bm25_canonical", "graph_canonical",
    "bm25_ranked", "graph_ranked", "oracle", "oracle_direct", "arithmetic_probe",
)
RETRIEVAL_NAMES = ("bm25_canonical", "graph_canonical", "bm25_ranked", "graph_ranked")

def parse_repository(sources):
    chunks, definitions, imports = {}, {}, {}
    for module, source in sources.items():
        tree = ast.parse(source)
        import_nodes = [node for node in tree.body if isinstance(node, ast.Import)]
        imports[module] = {alias.asname or alias.name: alias.name
                           for node in import_nodes for alias in node.names}
        import_lines = [ast.get_source_segment(source, node) for node in import_nodes]
        for function in (node for node in tree.body if isinstance(node, ast.FunctionDef)):
            key = f"{module}.{function.name}"
            definitions[key] = (module, function)
            body = ast.get_source_segment(source, function)
            chunks[key] = "\n".join([f"# {module}.py :: {function.name}", *import_lines, body])
    graph = {key: set() for key in chunks}
    for key, (module, function) in definitions.items():
        for call in (node.func for node in ast.walk(function) if isinstance(node, ast.Call)):
            destination = None
            if isinstance(call, ast.Name):
                destination = f"{module}.{call.id}"
            elif isinstance(call, ast.Attribute) and isinstance(call.value, ast.Name):
                imported = imports[module].get(call.value.id)
                if imported:
                    destination = f"{imported}.{call.attr}"
            if destination in graph:
                graph[key].add(destination)
    return chunks, graph

def terms(text):
    return re.findall(r"[a-z]+|\d+", text.lower())

def bm25_order(chunks, query, k1=1.5, b=0.75):
    documents = {key: Counter(terms(text)) for key, text in chunks.items() if key != ENTRY}
    df = Counter(term for doc in documents.values() for term in doc)
    lengths = {key: sum(doc.values()) for key, doc in documents.items()}
    avg_length = sum(lengths.values()) / len(lengths)
    query_terms = set(terms(query))
    scores = {}
    for key, document in documents.items():
        norm = k1 * (1 - b + b * lengths[key] / avg_length)
        scores[key] = math.fsum(
            math.log1p((len(documents) - df[term] + 0.5) / (df[term] + 0.5))
            * document[term] * (k1 + 1) / (document[term] + norm)
            for term in sorted(query_terms & document.keys())
        )
    return sorted(documents, key=lambda key: (-round(scores[key], 12), key))

def expand_order(ranked, graph, hops=3):
    distance = dict.fromkeys([ENTRY, ranked[0]], 0)
    frontier = set(distance)
    for depth in range(1, hops + 1):
        frontier = {neighbor for key in frontier for neighbor in graph[key]} - distance.keys()
        distance.update(dict.fromkeys(frontier, depth))
    return sorted(ranked, key=lambda key: distance.get(key, math.inf))

SEPARATOR = "\n\n"

def context_text(keys, chunks):
    return SEPARATOR.join(chunks[key] for key in keys)

def select_for_both_orders(ranked, chunks):
    selected = []
    for key in ranked:
        candidate = [*selected, key]
        if all(token_count(context_text(order, chunks)) <= RETRIEVAL_TOKEN_CAP
               for order in (candidate, sorted(candidate))):
            selected = candidate
    return selected

def make_prompt(case, context, direct=False):
    instruction = DIRECT_ASK if direct else TRACE_ASK
    return (
        f"Evaluate the exact result of compute({case['x']}). "
        "Module headers identify which file each function belongs to.\n"
        f"{instruction}\n\n"
        f"{case['chunks'][ENTRY]}\n\n{context}\n\nEnd of source."
    )

def build_conditions(case):
    chunks, graph = parse_repository(case["sources"])
    case["chunks"] = chunks
    required = case["trace"][1:]
    query = f"compute({case['x']})\n{chunks[ENTRY]}"
    ranked = bm25_order(chunks, query)
    distractors = sorted(key for key in ranked if key not in required)
    starts = {"front": 0, "middle": len(distractors) // 2, "end": len(distractors)}
    selected = {
        f"position_{name}": distractors[:start] + required + distractors[start:]
        for name, start in starts.items()
    }
    bm25 = select_for_both_orders(ranked, chunks)
    expanded = select_for_both_orders(expand_order(ranked, graph), chunks)
    selected.update({
        "bm25_canonical": sorted(bm25),
        "graph_canonical": sorted(expanded),
        "bm25_ranked": bm25,
        "graph_ranked": expanded,
        "oracle": required,
        "oracle_direct": required,
    })
    outputs = {}
    for name, keys in selected.items():
        context = context_text(keys, chunks)
        found = len(set(required) & set(keys))
        prompt = make_prompt(case, context, direct=name == "oracle_direct")
        outputs[name] = {
            "case_id": case["id"], "family": case["family"], "condition": name,
            "input": case["x"], "expected": case["answer"], "retrieved": keys,
            "trace": required, "expected_calls": case["calls"], "expected_events": case["events"],
            "coverage": found / len(required),
            "complete": found == len(required), "context_tokens": token_count(context),
            "prompt_tokens": token_count(prompt), "prompt": prompt,
            "first_evidence_fraction": (
                token_count(context[:context.index(chunks[required[0]])]) / token_count(context)
                if required[0] in keys else None
            ),
        }
    positions = [outputs[f"position_{name}"] for name in starts]
    assert len({row["prompt_tokens"] for row in positions}) == 1, "RQ1: prompt lengths differ."
    assert all(set(row["retrieved"]) == set(ranked) for row in positions)
    assert positions[0]["first_evidence_fraction"] == 0
    assert 0.35 < positions[1]["first_evidence_fraction"] < 0.65
    assert positions[2]["first_evidence_fraction"] > 0.75
    assert all(outputs[name]["context_tokens"] <= RETRIEVAL_TOKEN_CAP for name in RETRIEVAL_NAMES)
    for method in ("bm25", "graph"):
        assert set(selected[f"{method}_canonical"]) == set(selected[f"{method}_ranked"])
    assert outputs["oracle"]["complete"]
    first_fold = next(call for call in case["calls"] if call["function"] == "rounding_profile.fold")
    multiplier, addend, modulus = case["fold_coefficients"]
    expression = f"(({first_fold['arg']}) * {multiplier} + {addend}) % {modulus}"
    arithmetic_prompt = (
        "Evaluate this Python integer expression. Use Python's modulo rule. "
        'Return only one JSON object containing integer field "answer".\n'
        f"{expression}"
    )
    outputs["arithmetic_probe"] = {
        "case_id": case["id"], "family": case["family"], "condition": "arithmetic_probe",
        "input": first_fold["arg"], "expression": expression, "expected": first_fold["result"],
        "retrieved": [], "trace": [], "expected_calls": [], "expected_events": [],
        "coverage": None, "complete": None, "context_tokens": 0,
        "prompt_tokens": token_count(arithmetic_prompt), "prompt": arithmetic_prompt,
    }
    return list(outputs.values())

trials = [row for case in cases for row in build_conditions(case)]
assert len(trials) == len(cases) * len(CONDITION_NAMES)
assert len({(row["case_id"], row["condition"]) for row in trials}) == len(trials)
trial_order = list(trials)
random.Random(SEED + 1).shuffle(trial_order)

def table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

def percent(value):
    return f"{100 * value:.1f}%"

retrieval_names = (*RETRIEVAL_NAMES, "oracle")
retrieval_summary = []
for name in retrieval_names:
    group = [row for row in trials if row["condition"] == name]
    retrieval_summary.append((
        name, len(group), percent(sum(row["complete"] for row in group) / len(group)),
        percent(sum(row["coverage"] for row in group) / len(group)),
        f"{sum(row['context_tokens'] for row in group) / len(group):.1f}",
    ))

print(f"Đã kiểm tra {len(trials)} prompt. Dài nhất: {max(t['prompt_tokens'] for t in trials)} token nội dung.")
print(f"Giới hạn đoạn truy xuất: {RETRIEVAL_TOKEN_CAP} token theo {TOKENIZER}.")
table(["Cấu hình", "Số ca", "Tìm đủ hàm", "Độ bao phủ TB", "Token mã TB"], retrieval_summary)

## 3. Gọi API, chấm kết quả và ghi log

Notebook lấy khóa VTS từ biến môi trường VTS_LLM_API_KEY hoặc ô nhập ẩn. Nếu đổi URL, notebook dùng LLM_API_KEY hoặc ô nhập ẩn. Cấu hình xuất ra và các bảng kết quả không chứa khóa. Mã lượt chạy được xác định từ cấu hình và nội dung prompt; phản hồi đã lưu được dùng lại khi chạy tiếp.

Vết được chấm theo thứ tự sự kiện thực thi. Một giá trị trả về của hàm ngoài chỉ được so sau khi các lời gọi bên trong đã xảy ra. Điểm khớp chuỗi hàm, khớp đầy đủ vết và khớp đầu ra được tính riêng.

In [ ]:
def parse_payload(content):
    text = content.strip()
    fence = re.fullmatch(r"\x60\x60\x60(?:json)?\s*(.*?)\s*\x60\x60\x60", text, re.IGNORECASE | re.DOTALL)
    try:
        result = json.loads(fence.group(1) if fence else text)
    except (ValueError, TypeError):
        return None
    return result if isinstance(result, dict) else None

def parse_answer(content):
    result = parse_payload(content)
    if result and type(result.get("answer")) is int:
        return result["answer"]
    return None

assert parse_answer('{"answer": -9}') == -9
assert parse_answer('{"answer": true}') is None
assert parse_answer('7') is None

def score_response(trial, response):
    payload = parse_payload(response) or {}
    answer = payload.get("answer")
    if type(answer) is not int:
        answer = None
    predicted_calls = payload.get("calls")
    outcome = {"prediction": answer, "correct": answer == trial["expected"],
               "reported_calls": predicted_calls, "route_correct": None,
               "trace_correct": None, "first_error": None}
    if trial["condition"] in {"oracle_direct", "arithmetic_probe"}:
        return outcome

    gold = trial["expected_calls"]
    if not isinstance(predicted_calls, list) or any(not isinstance(call, dict) for call in predicted_calls):
        outcome.update(route_correct=False, trace_correct=False,
                       first_error={"kind": "missing_or_invalid_calls"})
        return outcome

    outcome["route_correct"] = (
        [call.get("function") for call in predicted_calls] == [call["function"] for call in gold]
    )
    for event, index in trial["expected_events"]:
        fields = ("function", "arg") if event == "call" else ("result",)
        actual = predicted_calls[index] if index < len(predicted_calls) else {}
        for field in fields:
            value = actual.get(field)
            expected = gold[index][field]
            if type(value) is not type(expected) or value != expected:
                kind = "missing_call" if not actual and index >= len(predicted_calls) else field
                outcome.update(trace_correct=False, first_error={
                    "kind": kind, "call_index": index, "function": gold[index]["function"],
                    "expected": expected, "observed": value,
                })
                return outcome

    if len(predicted_calls) > len(gold):
        outcome.update(trace_correct=False, first_error={"kind": "extra_call", "call_index": len(gold)})
    else:
        outcome["trace_correct"] = True
        if not outcome["correct"]:
            outcome["first_error"] = {
                "kind": "final_answer", "expected": trial["expected"], "observed": answer,
            }
    return outcome

def request_answer(prompt, api_key):
    payload = {
        "model": MODEL,
        "temperature": TEMPERATURE,
        OUTPUT_LIMIT_FIELD: MAX_OUTPUT_TOKENS,
        "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}],
    }
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    started = time.monotonic()
    for attempt in range(3):
        try:
            response = requests.post(
                CHAT_COMPLETIONS_URL, json=payload, headers=headers, timeout=90, allow_redirects=False,
            )
        except (requests.ConnectionError, requests.Timeout):
            if attempt == 2:
                raise
        else:
            if 200 <= response.status_code < 300:
                result = response.json()
                break
            if attempt == 2 or response.status_code not in {408, 429, 500, 502, 503, 504}:
                detail = response.text[:1000]
                if api_key:
                    detail = detail.replace(api_key, "[redacted]")
                raise RuntimeError(f"API HTTP {response.status_code}: {detail}")
        time.sleep(2 ** (attempt + 1))
    choice = result["choices"][0]
    content = choice["message"].get("content") or ""
    return {
        "response": content,
        "served_model": result.get("model"),
        "finish_reason": choice.get("finish_reason"),
        "usage": result.get("usage"),
        "attempts": attempt + 1,
        "latency_seconds": round(time.monotonic() - started, 3),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }

def save_csv(path, rows, fields):
    with path.open("w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def digest(data):
    return hashlib.sha256(json.dumps(data, sort_keys=True).encode()).hexdigest()

OUTPUT_DIR.mkdir(exist_ok=True)
dataset_path = OUTPUT_DIR / f"dataset-{digest(cases)[:12]}.json"
dataset_path.write_text(json.dumps(
    [{key: value for key, value in case.items() if key != "chunks"} for case in cases],
    indent=2, ensure_ascii=False,
), encoding="utf-8")

retrieval_path = OUTPUT_DIR / f"retrieval-{digest([cases, RETRIEVAL_TOKEN_CAP, TOKENIZER])[:12]}.csv"
save_csv(retrieval_path, (row for row in trials if row["condition"] in retrieval_names),
         ["case_id", "family", "condition", "coverage", "complete", "context_tokens", "retrieved", "trace"])

results = []
if RUN_LLM:
    if not CHAT_COMPLETIONS_URL.startswith(("https://", "http://")) or not MODEL.strip():
        raise ValueError("Điền CHAT_COMPLETIONS_URL (URL đầy đủ) và MODEL ở ô cấu hình rồi chạy lại.")
    key_variable = "VTS_LLM_API_KEY" if CHAT_COMPLETIONS_URL == VTS_CHAT_COMPLETIONS_URL else "LLM_API_KEY"
    if key_variable == "VTS_LLM_API_KEY":
        api_key = os.environ.get(key_variable) or getpass.getpass("VTS API key: ")
    else:
        api_key = os.environ.get(key_variable) or getpass.getpass("API key (Enter nếu endpoint không yêu cầu): ")
    if api_key and not CHAT_COMPLETIONS_URL.startswith("https://"):
        from urllib.parse import urlsplit
        if urlsplit(CHAT_COMPLETIONS_URL).hostname not in {"localhost", "127.0.0.1", "::1"}:
            raise ValueError("Dùng HTTPS khi gửi khóa API tới máy chủ từ xa.")
    configuration = {
        "model": MODEL, "url": CHAT_COMPLETIONS_URL, "system": SYSTEM, "temperature": TEMPERATURE,
        "output_limit": MAX_OUTPUT_TOKENS, "output_limit_field": OUTPUT_LIMIT_FIELD,
        "seed": SEED, "cases_per_family": CASES_PER_FAMILY, "tokenizer": TOKENIZER,
        "retrieval_token_cap": RETRIEVAL_TOKEN_CAP, "distractor_modules": DISTRACTOR_MODULES,
        "python_version": sys.version.split()[0],
        "tokenizer_version": importlib.metadata.version("tiktoken"),
    }
    run_id = digest([configuration, trials])[:12]
    (OUTPUT_DIR / f"config-{run_id}.json").write_text(json.dumps(configuration, indent=2), encoding="utf-8")
    log_path = OUTPUT_DIR / f"responses-{run_id}.jsonl"
    cache = {}
    if log_path.exists():
        for line in log_path.read_text(encoding="utf-8").splitlines():
            record = json.loads(line)
            cache[(record["case_id"], record["condition"])] = record
    with log_path.open("a", encoding="utf-8") as log:
        for index, trial in enumerate(trial_order, 1):
            key = (trial["case_id"], trial["condition"])
            if key not in cache:
                response = request_answer(trial["prompt"], api_key)
                record = {**trial, **response, **score_response(trial, response["response"])}
                log.write(json.dumps(record, ensure_ascii=False) + "\n")
                log.flush()
                cache[key] = record
            results.append(cache[key])
            if index == 1 or index % 10 == 0 or index == len(trial_order):
                print(f"{index}/{len(trial_order)} ca × cấu hình đã có kết quả.")
else:
    print("RUN_LLM = False. Đã lưu dữ liệu và kết quả truy xuất. Bảng độ chính xác cần chạy mô hình thật.")

## 4. Đọc kết quả

**RQ1:** So sánh theo ca vị trí đầu − giữa và cuối − giữa, ở cả chuỗi hàm và đầu ra.

**RQ2:** So sánh đồ thị − BM25 cùng thứ tự chuẩn để phân tích việc chọn đoạn. Sau đó so sánh thứ tự xếp hạng − thứ tự chuẩn trong từng phương pháp, trên đúng cùng tập đoạn.

**RQ3:** Kiểm tra các ca sai với oracle và phân bố bước sai đầu tiên. Đối chiếu phép thử số học riêng trên các ca oracle sai. Cột số học chỉ kiểm tra biểu thức fold đầu tiên, vì vậy kết quả không xác định nguyên nhân của các lỗi ở lời gọi sau.

Khoảng bootstrap 95% lấy mẫu ghép cặp trong mỗi họ và được diễn giải mô tả trên bộ ca tổng hợp nhỏ. Các bảng CSV và phản hồi gốc được lưu theo cùng mã lượt chạy.

In [ ]:
def summarize(rows, output_path):
    indexed = {(row["case_id"], row["condition"]): row for row in rows}
    case_ids = sorted({row["case_id"] for row in rows})
    assert len(rows) == len(indexed) == len(case_ids) * len(CONDITION_NAMES)

    def average(group, metric):
        values = [row[metric] for row in group if row[metric] is not None]
        return sum(values) / len(values) if values else None

    def show(value):
        return "—" if value is None else percent(value)

    statistics = []
    for condition in CONDITION_NAMES:
        group = [indexed[case_id, condition] for case_id in case_ids]
        statistics.append({
            "condition": condition, "n": len(group), "accuracy": average(group, "correct"),
            "route_accuracy": average(group, "route_correct"), "trace_accuracy": average(group, "trace_correct"),
            "complete_retrieval": average(group, "complete"),
            "invalid_answers": sum(row["prediction"] is None for row in group),
        })
    table(["Cấu hình", "n", "Đúng đầu ra", "Đúng chuỗi gọi", "Đúng cả vết", "Tìm đủ hàm", "Sai định dạng"], [
        (row["condition"], row["n"], show(row["accuracy"]), show(row["route_accuracy"]),
         show(row["trace_accuracy"]), show(row["complete_retrieval"]), row["invalid_answers"])
        for row in statistics
    ])

    def paired(label, chosen, baseline, measure):
        strata = defaultdict(list)
        for case_id in case_ids:
            a, b = indexed[case_id, chosen], indexed[case_id, baseline]
            strata[a["family"]].append(int(a[measure]) - int(b[measure]))
        delta = [x for group in strata.values() for x in group]
        rng = random.Random(SEED + 31)
        samples = sorted(
            sum(sum(rng.choices(group, k=len(group))) for group in strata.values()) / len(delta)
            for _ in range(2000)
        )
        return {"comparison": label, "treatment": chosen, "reference": baseline, "metric": measure,
                "n": len(delta), "improved": delta.count(1), "worsened": delta.count(-1),
                "delta": sum(delta) / len(delta), "ci_low": samples[49], "ci_high": samples[1949]}

    contrasts = [
        ("RQ1 đầu − giữa", "position_front", "position_middle", ("route_correct", "correct")),
        ("RQ1 cuối − giữa", "position_end", "position_middle", ("route_correct", "correct")),
        ("RQ2 chọn đoạn", "graph_canonical", "bm25_canonical", ("complete", "route_correct", "correct")),
        ("RQ2 thứ tự BM25", "bm25_ranked", "bm25_canonical", ("route_correct", "correct")),
        ("RQ2 thứ tự đồ thị", "graph_ranked", "graph_canonical", ("route_correct", "correct")),
        ("RQ3 đối chứng prompt oracle", "oracle", "oracle_direct", ("correct",)),
    ]
    comparisons = [
        paired(label, chosen, baseline, metric)
        for label, chosen, baseline, metrics in contrasts for metric in metrics
    ]
    table(["Phép so sánh", "Chỉ số", "Chênh (điểm %)", "Bootstrap 95% (điểm %)", "Tăng", "Giảm"], [
        (r["comparison"], r["metric"], f"{100*r['delta']:+.1f}",
         f"[{100*r['ci_low']:+.1f}, {100*r['ci_high']:+.1f}]", r["improved"], r["worsened"])
        for r in comparisons
    ])

    diagnostic = []
    for case_id in case_ids:
        oracle = indexed[case_id, "oracle"]
        probe = indexed[case_id, "arithmetic_probe"]
        error = oracle["first_error"] or {}
        diagnostic.append({
            "case_id": case_id, "family": oracle["family"], "oracle_correct": oracle["correct"],
            "oracle_route_correct": oracle["route_correct"], "oracle_trace_correct": oracle["trace_correct"],
            "first_error": error.get("kind"), "at_function": error.get("function"),
            "call_index": error.get("call_index"), "expected_at_error": error.get("expected"),
            "observed_at_error": error.get("observed"), "oracle_expected": oracle["expected"],
            "oracle_prediction": oracle["prediction"], "probe_expression": probe["expression"],
            "probe_expected": probe["expected"], "probe_prediction": probe["prediction"],
            "probe_correct": probe["correct"],
        })
    failures = [row for row in diagnostic if not row["oracle_correct"]]
    buckets = Counter(row["first_error"] or "unknown" for row in failures)
    print("Bước sai đầu tiên trên ca oracle sai đầu ra:", dict(buckets))
    print("Phép thử số học đúng trên ca oracle sai:", sum(r["probe_correct"] for r in failures), "/", len(failures))

    oracle_pairs = []
    for method in ("bm25_canonical", "graph_canonical"):
        for complete in (False, True):
            failed = [indexed[cid, method] for cid in case_ids
                      if indexed[cid, method]["complete"] == complete and not indexed[cid, method]["correct"]]
            fixed = sum(indexed[row["case_id"], "oracle"]["correct"] for row in failed)
            oracle_pairs.append({"condition": method, "complete_retrieval": complete, "wrong": len(failed),
                                 "fixed_with_oracle": fixed, "still_wrong": len(failed) - fixed})
    table(["RQ3 cấu hình", "Tìm đủ hàm", "Sai", "Oracle sửa được", "Oracle vẫn sai"], [
        (r["condition"], "Có" if r["complete_retrieval"] else "Thiếu", r["wrong"],
         r["fixed_with_oracle"], r["still_wrong"]) for r in oracle_pairs
    ])
    for name, records in (("summary", statistics), ("paired", comparisons),
                          ("diagnostics", diagnostic), ("oracle-pairs", oracle_pairs)):
        save_csv(output_path / f"{name}-{run_id}.csv", records, list(records[0]))

if results:
    summarize(results, OUTPUT_DIR)

print(f"Thư mục kết quả: {OUTPUT_DIR.resolve()}")